# Week 6 — Testing the Mirage

**LLMs & You · Hampden-Sydney College · Fall 2026**

Tuesday we argued about whether chain of thought is reasoning. Today you test it
yourself, twice:

1. **On the big models, in BoodleBox.** Catch a model saying something false
   with confidence — a *hallucination* — then find out whether the other models
   make the same mistake, and whether asking them to think step by step fixes it.
2. **On the small models, here in Colab.** Take the two models you used in
   Weeks 3 and 4, give them math word problems with and without a chain-of-thought
   prompt, and measure whether the chain helps.

The two halves are one experiment. Wei et al. said chain of thought only pays
off in very large models; the mirage paper said it only pays off on questions
like the ones the model practiced. Part 1 is the large models. Part 2 is the
small ones. Bring both to Thursday.

---

## This is not a worksheet

Nothing to hand in on Wednesday night. Work through what interests you, change
the inputs, and find **one thing worth showing on Thursday** — ideally something
that broke or surprised you. Part 1 needs no code at all.

Part 2 runs in Google Colab in your browser. Nothing to install, no account
beyond Google, no API key, nothing to pay for.

---
# Part 1 — Hunting a hallucination in BoodleBox

This part happens in [BoodleBox](https://boodlebox.ai), on the college's
subscription, not in this notebook. If you do not have an account yet, the
[BoodleBox guide](https://nalaquq.github.io/llms-and-you/guides/boodlebox/) says
how to get one. The notebook is here to tell you what to do and to give you
somewhere tidy to keep notes.

## 1. What you are hunting

A **hallucination** is a fluent, confident answer that is false, or that cites
something that does not exist. Not a refusal, not a vague answer — a specific
claim, stated as fact, that you can show is wrong.

Week 7 is about why models do this. Today you just need to catch one, which is
harder than it sounds: the models on BoodleBox are large and recent, and they
are right about most things most people ask. You have to go looking.

Places they are known to slip:

| Try asking for… | Why it tends to go wrong |
|:---|:---|
| **Sources.** “List three peer-reviewed papers on *[a narrow topic]*, with authors, year and DOI.” | A reference is a pattern — authors, year, journal — easy to produce whether or not the paper exists |
| **Local or obscure facts.** Hampden-Sydney history, a small town, a minor historical figure | Little written about it, so little in the training data to be right from |
| **A question with a false premise.** “Why did *[real person]* win *[a prize they never won]*?” | The model tends to answer the question it was asked rather than challenge it |
| **Something that does not exist.** “Summarize the 2019 paper *[made-up title]* by *[made-up authors]*.” | Same pattern-completion, with nothing true to find |
| **Exact details.** A quotation with its page number, a statistic with its source, the date of a small event | The general shape is right; the specifics are invented to fit it |
| **Anything very recent** | It happened after the model's training data was collected |

Two rules. **Only count it if you can check it** — name the source that shows
the answer is wrong (a library database, the college website, the book itself).
And **this is about false facts, not harmful content**: don't try to get a
model to say something dangerous. That is a different exercise and not this one.

## 2. Catch one, then try the others

1. Pick one model in BoodleBox and hunt until you get a hallucination you can
   prove. Write down the **exact model name** BoodleBox shows, and **copy your
   prompt word for word**.
2. **Start a new chat** for every attempt. A model can see earlier messages in
   the same chat, and that changes its answer.
3. Paste the *identical* prompt into each of the other models BoodleBox offers
   you. Record what each one says.
4. Run your original prompt a second time on the first model. Chat tools usually
   sample with some temperature (Tuesday, slide 23), so the same prompt can come
   back different. Does the hallucination come back?

Then decide what you think happened. Some starting points — pick one, or argue
for your own:

- Was the true answer **rare** in anything the model could have read?
- Did the prompt carry a **false premise** the model accepted?
- Did the models give the **same** wrong answer, or **different** ones? The same
  wrong answer across companies hints at a shared source; different ones hint
  that each is guessing.
- Tuesday's mirage paper said a model “writes steps that look like steps it has
  seen”. Did your model write *an answer that looks like an answer*?

## 3. Now add a chain of thought

Take your hallucination prompt back to the model that produced it — in a **new
chat** — and add a chain-of-thought instruction to the end. Try two versions:

> *…your prompt…* **Let's think step by step.**

> *…your prompt…* **Think step by step. For each step, say how you know it, and
> say so if you are not sure.**

For each model, which of these happened?

- **Fixed.** The chain caught the error and the answer is now right.
- **Flagged.** The answer is hedged, or the model now says it does not know.
- **Unchanged.** Same false claim, now with steps in front of it.
- **Worse.** A new false claim, or a longer and more convincing wrong answer —
  what the mirage paper called “fluent nonsense”.

And the question Tuesday left open: **when the chain fixed it, can you see
where in the chain the fix happened?** When it didn't, does the chain contain the
mistake, or does the chain look fine and the answer is still wrong?

### Keep your notes here (optional)

The cell below does not talk to BoodleBox — it is a table for your own notes, so
your Thursday write-up is easy to put together. Replace the example row with
yours, add a row per model, and run it. Skip it if you would rather keep notes
somewhere else.

In [ ]:
import pandas as pd

hallucination_log = pd.DataFrame([
    {
        "model": "example: the exact name BoodleBox shows",
        "prompt": "copied word for word",
        "what it said": "the false claim, quoted",
        "what is true": "the correct answer",
        "how you checked": "the source you used",
        "same mistake as the first model?": "yes / no / different mistake",
        "with 'Let's think step by step'": "fixed / flagged / unchanged / worse",
    },
    # add one row per model you tried
])
hallucination_log

---
# Part 2 — Chain of thought on the models from Weeks 3 and 4

Now the other end of the scale. You have met both of these before:

| model | from | size | trained to follow instructions? |
|:---|:---|:---|:---|
| `google/flan-t5-base` | Week 3, *Watching Attention Work* | 248 million parameters | yes |
| `distilgpt2` | Week 4, *Fine-Tuning a Language Model* | 82 million parameters | no — it only continues text |

The models on BoodleBox are far bigger — their makers mostly no longer publish
exact sizes. On Tuesday, Wei et al.'s Figure 4 showed chain of thought making every model under about ten billion
parameters *worse* at math word problems. Here is your chance to check.

## 4. Load both models

One cell, once per session. Colab forgets everything when the tab is closed. It
downloads about 1.3 GB the first time.

In [ ]:
# Colab already has torch and pandas. transformers and sentencepiece it may not.
%pip install --quiet "transformers>=4.40" sentencepiece

import re

import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

MODELS = {}


def load(name):
    """Load a model once and keep it. Seq2seq for flan-t5, causal for gpt-2."""
    if name not in MODELS:
        tokenizer = AutoTokenizer.from_pretrained(name)
        kind = AutoModelForSeq2SeqLM if "t5" in name else AutoModelForCausalLM
        MODELS[name] = (tokenizer, kind.from_pretrained(name).eval())
    return MODELS[name]


def ask(name, prompt, max_new_tokens=100):
    """Prompt in, text out. Greedy decoding: temperature zero, same answer every run."""
    tokenizer, model = load(name)
    enc = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    if model.config.is_encoder_decoder:
        return tokenizer.decode(out[0], skip_special_tokens=True)
    # A causal model echoes the prompt back; keep only what it added.
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)


for name in ["google/flan-t5-base", "distilgpt2"]:
    load(name)
    print("loaded", name)

## 5. The problems, and the two prompts

Six math word problems with known answers. Two are famous — the tennis balls
and the cafeteria are Wei et al.'s own Figure 1 examples, which means they have
been all over the internet since 2022 (Tuesday, slide 21: data leakage). The
other four were written for this notebook.

Each problem is asked two ways:

- **direct** — *answer with a number only*
- **chain of thought** — the zero-shot cue from Tuesday, *Let's think step by step.*

The answer is marked the simplest way possible: take the **last number** the
model wrote and check it against the right answer. That is exact match on the
answer only — Tuesday, slide 24 — and it ignores whether the steps were right.

In [ ]:
PROBLEMS = [
    ("The cafeteria had 23 apples. They used 20 to make lunch and bought 6 more. "
     "How many apples do they have?", 9),
    ("Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has "
     "3 tennis balls. How many tennis balls does he have now?", 11),
    ("A bus has 14 people on it. At the first stop 5 people get off and 3 get on. "
     "At the second stop 4 people get off. How many people are on the bus now?", 8),
    ("Maria has 3 boxes with 12 pencils in each box. She gives away 10 pencils. "
     "How many pencils does she have left?", 26),
    ("Tom has 7 marbles and finds 5 more. How many marbles does he have?", 12),
    ("A shirt costs 15 dollars. A pair of jeans costs twice as much as the shirt. "
     "How much do the shirt and the jeans cost together?", 45),
]

DIRECT = "Answer with a number only.\nQ: {question}\nA:"
CHAIN_OF_THOUGHT = "Q: {question}\nA: Let's think step by step."


def last_number(text):
    """The last number in the text, or None. Crude on purpose -- see the questions below."""
    found = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(found[-1]) if found else None


def compare(name, problems=PROBLEMS, templates=None, show=True):
    """Ask every problem every way; return one row per problem."""
    templates = templates or {"direct": DIRECT, "chain of thought": CHAIN_OF_THOUGHT}
    rows = []
    for question, answer in problems:
        row = {"question": question[:45] + "…", "answer": answer}
        for label, template in templates.items():
            said = ask(name, template.format(question=question))
            row[label] = last_number(said)
            row[label + " right?"] = last_number(said) == answer
            if show:
                print(f"[{label}] {question[:60]}…\n    {said!r}\n")
        rows.append(row)
    table = pd.DataFrame(rows)
    for label in templates:
        print(f"{name} — {label}: {table[label + ' right?'].sum()} of {len(table)} right")
    return table

## 6. Predict, then run

**Before you run the next cell**, write down how many of the six you think
`flan-t5-base` gets right each way. Then run it and read every answer, not just
the score at the bottom.

In [ ]:
flan_results = compare("google/flan-t5-base")
flan_results

!!! tip "Your turn"

    **Read the chains, not the score.** Pick one the model got wrong with chain of
    thought and find the exact step where it goes off the rails. Is the mistake in
    the arithmetic (*7 + 5 = 10*), or in understanding the story (adding what
    should be taken away)?

    Notice that `flan-t5-base` often writes steps even when you asked for a number
    only. It was trained on examples of chain-of-thought answers, so that is the
    habit it learned. Tuesday's word for this: it writes steps that *look like*
    steps it has seen.

    Wei et al. wrote that smaller models “produced fluent but illogical chains of
    thought”. Is that what you are looking at?

## 7. The model that never learned to follow instructions

`distilgpt2` was trained only to continue text — no examples of questions and
answers, no instructions. Predict what a chain-of-thought prompt does to a model
like that, then run it.

In [ ]:
gpt_results = compare("distilgpt2")
gpt_results

!!! tip "Your turn"

    What did it actually do with *Let's think step by step*? A chain-of-thought
    prompt assumes a model that knows what a question is and what an answer looks
    like. Where does that assumption come from, and which of Tuesday's slides
    already told you this would happen?

    Look at the scores, too. If `distilgpt2` got anything “right”, was it right, or
    did the last-number rule happen to find the answer's number somewhere in the
    noise? That is the weakness of marking by a crude rule. What would a better
    rule be?

---
## 8. Change the experiment

Everything below is yours to edit. Three directions worth trying — pick one.

**Show it a worked example.** Wei et al. did not use *Let's think step by step*;
they put worked examples in front of the question (few-shot chain of thought,
Tuesday slide 5). The cell below does that with Wei's own tennis-ball example.
Does a worked example help more than the one-line cue?

In [ ]:
WORKED_EXAMPLE = (
    "Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has "
    "3 tennis balls. How many tennis balls does he have now?\n"
    "A: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. "
    "5 + 6 = 11. The answer is 11.\n\n"
)

few_shot = compare(
    "google/flan-t5-base",
    problems=PROBLEMS[2:],  # leave out the two famous ones, including the example itself
    templates={
        "zero-shot chain of thought": CHAIN_OF_THOUGHT,
        "few-shot chain of thought": WORKED_EXAMPLE + "Q: {question}\nA:",
    },
    show=False,
)
few_shot

**Write your own problems.** Add to the list below and rerun. Try moving along the
mirage paper's three axes (Tuesday, slide 20): a problem with more steps than
these (*length*), a kind of problem these don't include, like percentages or
time (*task*), or the same problem set out oddly, with a stray word or the
question first (*format*). Does chain of thought help more, less, or not at all as
you move away from simple two-step problems?

In [ ]:
MY_PROBLEMS = [
    ("A class has 20 students. A quarter of them are absent. How many are present?", 15),
    # (your question, the right answer),
]

mine = compare("google/flan-t5-base", problems=MY_PROBLEMS)
mine

**Try a bigger model.** `flan-t5-large` is the same family at 780 million
parameters — three times the size, and still about eighty times smaller than
PaLM 62B, the smallest model where Wei et al. saw chain of thought clearly pay off. It downloads about 3 GB, so give it a
couple of minutes. Does three times the size change the score? Would you expect
it to, after Tuesday's slide 13?

In [ ]:
large_results = compare("google/flan-t5-large", show=False)
large_results

---
## What to bring on Thursday

Two short reports and one finding.

**From Part 1 — the hallucination.**

- What was it? The prompt, the model, the false claim, and the source that shows
  it is false.
- Which other models made the same mistake, which made a different one, and
  which got it right?
- Why do you think it happened? One reason, with the evidence for it.
- What did adding a chain of thought do — fixed, flagged, unchanged or worse?

**From Part 2 — chain of thought on small models.**

- Did chain of thought improve the answers? Give the scores, and one chain you
  read closely.
- Does what you saw fit Wei et al.'s claim that chain of thought only helps in
  large models?

**The finding, in three sentences:**

> *I expected X. I ran Y. I got Z, which surprised me because…*

A method that broke is worth more than one that worked — the labs are graded on
the write-up, not the result. A hallucination you could *not* produce is a result
too, if you say what you tried. A confusion you can state precisely counts.

<small>Questions before Thursday: sgleason@hsc.edu, or office hours T/Th 12–2.</small>